In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm

# Executive Summary: WA DOH Food Safety Risk & Resource Allocation Audit

---

## 1. Executive Context & Objective
The Washington State Department of Health (WA DOH) oversees food safety and environmental public health across local health jurisdictions (LHJs). However, varying county budgets, staffing levels, and commercial density create significant regional disparities in inspection capacity.

This project built an **automated data analysis pipeline** to evaluate operational capacity, identify severe inspection backlogs, and map public health risk drivers across all Washington county health jurisdictions. The primary objective is to transition state resource allocation from reactive outbreak response to **data-driven, proactive risk mitigation**.

---

## 2. Composite Risk Scoring Framework

> **Composite Risk Score (0–100 Scale)** combines three core weighted operational and epidemiological factors to quantify county risk:

* **Inspection Coverage Deficit (40% Weight):** Measures systemic monitoring gaps in permanent food service establishments.
* **Normalized Backlog Count (30% Weight):** Captures raw volume workload stress (permitted establishments minus actual inspections conducted).
* **Outbreak Severity Index (30% Weight):** Measures active disease outbreak investigations (*Salmonella*, *E. coli*, norovirus).

---

## 3. Strategic Action Matrix

| Risk Tier | Condition | Priority Action Plan | Resource Allocation |
| :--- | :--- | :--- | :--- |
| 🔴 **RED ALERT** | Coverage $< 80\%$ **AND** Outbreaks $\ge 1$ | **DEPLOY EMERGENCY INSPECTORS:** Immediate state audit, deployment of relief inspectors, and active outbreak tracing support. | **Direct State Intervention** |
| 🟡 **YELLOW WARNING** | Backlog $> 30$ kitchens **OR** Coverage $< 85\%$ | **TARGETED STAFFING SUPPORT:** Authorize overtime funding, streamline local inspection scheduling, and reallocate regional staff. | **Targeted Financial & Staffing Support** |
| 🟢 **GREEN** | Operational targets met | **MAINTAIN STANDARD OPERATIONS:** Maintain standard oversight; document and share local operational best practices. | **Standard Oversight** |

---

## 4. Operational Deliverables Generated

1. `output/lhj_risk_summary_report.csv`  
   *Ranked executive dashboard listing all jurisdictions by Composite Risk Score, Risk Tier, and Recommended State Action.*
2. `output/lhj_cleaned_full_dataset.csv`  
   *Normalized data layer containing all cleaned metrics ready for integration into state Tableau dashboards or GIS mapping software.*

In [4]:
df_raw=pd.read_csv("export.csv")

In [5]:
df_raw.head(10)

,Entity,Environmental Public Health - Food Safety: Permanent food service establishments permitted,Environmental Public Health - Food Safety: Permanent food service establishments inspected,Environmental Public Health - Food Safety: Routine permanent food service establishment inspections,Environmental Public Health - Food Safety: Temporary food service establishments permitted,Environmental Public Health - Food Safety: Temporary food service establishments inspected,Environmental Public Health - Food Safety: Temporary food service establishment inspections,Environmental Public Health - Food Safety: Food worker cards issued,Environmental Public Health - Food Safety: Food service establishment complaints investigated,Environmental Public Health - Food Safety: Investigations of reported foodborne/waterborne disease cases,...,Prevention & Promotion - Tobacco Prevention and Control: Compliance inspections/investigations conducted,Prevention & Promotion - Tobacco Prevention and Control: Citations/fines issued for violations,Prevention & Promotion - Injury Prevention: Is your LHJ involved in initiatives to decrease the rate of injury the community,Other Prevention & Promotion Initiatives: LHJ addressed other health risks through prevention and wellness initiatives - Asthma,Other Prevention & Promotion Initiatives: LHJ addressed other health risks through prevention and wellness initiatives - HIV/AIDS,Other Prevention & Promotion Initiatives: LHJ addressed other health risks through prevention and wellness initiatives - Sexual health/STDs,Other Prevention & Promotion Initiatives: LHJ addressed other health risks through prevention and wellness initiatives - Violence,Other Prevention & Promotion Initiatives: LHJ addressed other health risks through prevention and wellness initiatives - Substance abuse,Other Prevention & Promotion Initiatives: LHJ addressed other health risks through prevention and wellness initiatives - Mental health/ - Behavioral health,Other Prevention & Promotion Initiatives: LHJ addressed other health risks through prevention and wellness initiatives - Land use and/ - or transportation plans
0,Washington State Department of Health,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Adams County Health Department,125,125,248,148,134,134,917,5,10,...,0.0,0.0,No,No,No,No,Yes,No,Yes,No
2,Asotin County Health District,79,22,28,18,18,18,652,1,1,...,0.0,0.0,No,No,No,No,No,No,No,No
3,Benton-Franklin Health District,"1,409","1,407","3,138","1,036","1,012","1,023","10,116",266,47,...,3.0,0.0,Yes,No,Yes,Yes,Yes,Yes,Yes,Yes
4,Chelan-Douglas Health District,710,698,993,438,256,282,"6,776",32,22,...,8.0,0.0,No,No,No,Yes,No,No,No,Yes
5,Clallam County Department of Health & Human Se...,516,516,536,204,177,177,"3,256",20,8,...,NaN,NaN,No,No,Yes,Yes,No,Yes,Yes,No
6,Clark County Public Health,"1,514","1,597","3,041",239,239,215,"21,171",211,127,...,7.0,0.0,Yes,No,Yes,Yes,Yes,Yes,No,Yes
7,Columbia County Public Health Department,32,32,33,60,43,43,166,0,0,...,NaN,0.0,Yes,No,No,No,No,Yes,No,No
8,Cowlitz County Health & Human Services Department,585,585,971,298,298,329,"5,163",110,26,...,0.0,0.0,No,Yes,No,No,No,No,Yes,Yes
9,Garfield County Health District,17,17,27,12,12,15,73,3,0,...,3.0,0.0,No,No,Yes,Yes,No,No,No,Yes


In [ ]:
df.shape

(38, 229)

Project Agenda: To build a data-driven risk assessment pipeline that identifies high-risk health jurisdictions, pinpoints food safety inspection backlogs, and optimizes the allocation of state public health resources to prevent foodborne disease outbreaks.

Core Objectives Breakdown
Quantify Operational Capacity & Backlogs

Determine which counties have a high volume of food permits but low inspection coverage (e.g., identifying uninspected kitchens).

Correlate Backlogs with Public Health Risks

Measure if uninspected establishments and high complaint rates directly lead to an increase in formal foodborne/waterborne disease investigations.

Establish a Priority Risk Index (Red / Yellow / Green)

Categorize all Washington state counties into clear risk tiers based on coverage percentage, backlog count, and disease outbreak activity.

Deliver Actionable Decisions for Health Directors

Provide state leadership with a clear roadmap on where to deploy emergency inspectors, where to allocate state grant funding, and where to expand public health initiatives (such as tobacco control, asthma, and behavioral health).

In [7]:
import os
import pandas as pd


def run_lhj_risk_pipeline(data_file_path):
    # Define output directory
    output_dir = "data_exports"

    # --------------------------------------------------------------------------
    # STEP 1: LOAD DATA
    # --------------------------------------------------------------------------
    df_raw = pd.read_csv(data_file_path)
    print(f"[STEP 1] Raw data loaded. Shape: {df_raw.shape}")

    # --------------------------------------------------------------------------
    # STEP 2: RENAME COLUMNS TO SIMPLE, EASY-TO-UNDERSTAND NAMES
    # --------------------------------------------------------------------------
    column_mapping = {
        df_raw.columns[0]: "county_name",
        df_raw.columns[1]: "perm_permitted",
        df_raw.columns[2]: "perm_inspected",
        df_raw.columns[3]: "perm_routine_inspections",
        df_raw.columns[4]: "temp_permitted",
        df_raw.columns[5]: "temp_inspected",
        df_raw.columns[6]: "temp_routine_inspections",
        df_raw.columns[7]: "food_worker_cards",
        df_raw.columns[8]: "complaints_investigated",
        df_raw.columns[9]: "foodborne_investigations",
        df_raw.columns[10]: "enforcement_reinspections",
        df_raw.columns[11]: "tobacco_inspections",
        df_raw.columns[12]: "tobacco_fines_issued",
        df_raw.columns[13]: "has_injury_prevention_prog",
    }

    # Apply column renaming
    df = df_raw.rename(columns=column_mapping)
    print("[STEP 2] Column names successfully simplified.")

    # --------------------------------------------------------------------------
    # STEP 3: DATA CLEANING & ROW CUTTING
    # --------------------------------------------------------------------------
    # FIX 1: Dynamically drop "State Total" / state summary rows to prevent data leakage
    state_rows = df["county_name"].astype(str).str.contains(
        "State Total|Washington State|State Summary", case=False, na=False
    )
    df = df[~state_rows].reset_index(drop=True)
    print(
        f"         State summary row(s) dropped. Cleaned Shape: {df.shape}"
    )

    # Clean string formatted numbers (e.g., '1,409' -> 1409.0)
    string_cols = df.select_dtypes(include=["object"]).columns.drop(
        "county_name"
    )

    for col in string_cols:
        cleaned_series = (
            df[col].astype(str).str.replace(",", "", regex=True).str.strip()
        )
        numeric_converted = pd.to_numeric(cleaned_series, errors="coerce")

        if (
            numeric_converted.notna().sum() > 0
            or cleaned_series.isin(["nan", "NaN"]).all()
        ):
            df[col] = numeric_converted

    # Fill missing numerical food safety values with 0
    core_numeric_cols = [
        "perm_permitted",
        "perm_inspected",
        "perm_routine_inspections",
        "temp_permitted",
        "temp_inspected",
        "temp_routine_inspections",
        "food_worker_cards",
        "complaints_investigated",
        "foodborne_investigations",
        "enforcement_reinspections",
    ]

    for col in core_numeric_cols:
        if col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(0)

    print("         Comma removal and numeric conversion complete.")

    # FIX 2: Defragment memory to prevent PerformanceWarnings when adding new calculated columns
    df = df.copy()

    # --------------------------------------------------------------------------
    # STEP 4: METRIC CALCULATION
    # --------------------------------------------------------------------------
    # A. Operational Ratios
    df["perm_coverage_pct"] = (
        df["perm_inspected"] / (df["perm_permitted"] + 1e-5)
    ) * 100
    df["perm_backlog_count"] = df["perm_permitted"] - df["perm_inspected"]

    df["temp_coverage_pct"] = (
        df["temp_inspected"] / (df["temp_permitted"] + 1e-5)
    ) * 100
    df["temp_backlog_count"] = df["temp_permitted"] - df["temp_inspected"]

    # B. Public Health Risk Ratios
    df["outbreak_escalation_ratio"] = df["foodborne_investigations"] / (
        df["complaints_investigated"] + 1
    )

    # C. Composite Risk Score (0 - 100 Scale)
    max_backlog = (
        df["perm_backlog_count"].max()
        if df["perm_backlog_count"].max() > 0
        else 1
    )
    max_outbreaks = (
        df["foodborne_investigations"].max()
        if df["foodborne_investigations"].max() > 0
        else 1
    )

    norm_backlog = df["perm_backlog_count"] / max_backlog
    norm_outbreaks = df["foodborne_investigations"] / max_outbreaks
    coverage_deficit = (100 - df["perm_coverage_pct"]) / 100

    df["composite_risk_score"] = (
        (0.4 * coverage_deficit) + (0.3 * norm_backlog) + (0.3 * norm_outbreaks)
    ) * 100
    df["composite_risk_score"] = df["composite_risk_score"].round(2)

    print(
        "[STEP 4] Operational metrics and Risk Scores successfully calculated."
    )

    # --------------------------------------------------------------------------
    # STEP 5: RISK CATEGORIZATION & DECISION RULES
    # --------------------------------------------------------------------------
    def assign_risk_category(row):
        if row["perm_coverage_pct"] < 80 and row["foodborne_investigations"] > 0:
            return "RED ALERT"
        elif row["perm_backlog_count"] > 30 or row["perm_coverage_pct"] < 85:
            return "YELLOW WARNING"
        else:
            return "GREEN"

    def assign_action(row):
        if row["risk_tier"] == "RED ALERT":
            return (
                "DEPLOY EMERGENCY INSPECTORS: Immediate state audit & outbreak"
                " control"
            )
        elif row["risk_tier"] == "YELLOW WARNING":
            return (
                "ALLOCATE OVERTIME / STAFF: Target high backlog before"
                " outbreaks occur"
            )
        else:
            return "MAINTAIN REGULAR OPERATIONS: Share operational best practices"

    df["risk_tier"] = df.apply(assign_risk_category, axis=1)
    df["recommended_action"] = df.apply(assign_action, axis=1)

    print("[STEP 5] County Risk Tiers and Decision Matrix assigned.")

    # --------------------------------------------------------------------------
    # STEP 6: OUTPUT GENERATION & REPORTING
    # --------------------------------------------------------------------------
    os.makedirs(output_dir, exist_ok=True)

    executive_cols = [
        "county_name",
        "risk_tier",
        "composite_risk_score",
        "perm_permitted",
        "perm_inspected",
        "perm_backlog_count",
        "perm_coverage_pct",
        "foodborne_investigations",
        "complaints_investigated",
        "recommended_action",
    ]

    summary_df = df[executive_cols].sort_values(
        by="composite_risk_score", ascending=False
    )

    # Export CSVs
    summary_df.to_csv(
        os.path.join(output_dir, "lhj_risk_summary_report.csv"), index=False
    )
    df.to_csv(
        os.path.join(output_dir, "lhj_cleaned_full_dataset.csv"), index=False
    )

    # Executive Summary Prints
    print("\n" + "=" * 70)
    print("EXECUTIVE RISK SUMMARY RESULTS")
    print("=" * 70)

    print(f"\nTotal Health Jurisdictions Analyzed: {len(df)}")
    print(
        "  🔴 Red Alert Jurisdictions:   "
        f" {(df['risk_tier'] == 'RED ALERT').sum()}"
    )
    print(
        "  🟡 Yellow Warning Jurisdictions:"
        f" {(df['risk_tier'] == 'YELLOW WARNING').sum()}"
    )
    print(
        "  🟢 Green Jurisdictions:         "
        f" {(df['risk_tier'] == 'GREEN').sum()}"
    )

    print("\n--- TOP 5 HIGHEST RISK JURISDICTIONS ---")
    top_5 = summary_df.head(5)[
        [
            "county_name",
            "risk_tier",
            "composite_risk_score",
            "perm_backlog_count",
            "foodborne_investigations",
        ]
    ]
    print(top_5.to_string(index=False))

    print(
        f"\n[SUCCESS] Reports exported to folder: '{os.path.abspath(output_dir)}'"
    )

    return df, summary_df


# ==============================================================================
# EXECUTION ENTRY POINT
# ==============================================================================
if __name__ == "__main__":
    try:
        full_df, summary_df = run_lhj_risk_pipeline("export.csv")
    except FileNotFoundError as e:
        print(f"\n[ERROR] {e}")

[STEP 1] Raw data loaded. Shape: (38, 229)
[STEP 2] Column names successfully simplified.
         State summary row(s) dropped. Cleaned Shape: (36, 229)
         Comma removal and numeric conversion complete.
[STEP 4] Operational metrics and Risk Scores successfully calculated.
[STEP 5] County Risk Tiers and Decision Matrix assigned.

EXECUTIVE RISK SUMMARY RESULTS

Total Health Jurisdictions Analyzed: 36
  🔴 Red Alert Jurisdictions:    4
  🟡 Yellow Warning Jurisdictions: 11
  🟢 Green Jurisdictions:          21

--- TOP 5 HIGHEST RISK JURISDICTIONS ---
                             county_name      risk_tier  composite_risk_score  perm_backlog_count  foodborne_investigations
        Whatcom County Health Department      RED ALERT                 50.00               574.0                       1.0
Wahkiakum County Health & Human Services YELLOW WARNING                 40.00                 0.0                       0.0
                         Multiple County YELLOW WARNING             

In [ ]:
df.head()

,county_name,Environmental Public Health - Food Safety: Permanent food service establishments permitted,Environmental Public Health - Food Safety: Permanent food service establishments inspected,perm_routine_inspections,Environmental Public Health - Food Safety: Temporary food service establishments permitted,Environmental Public Health - Food Safety: Temporary food service establishments inspected,Environmental Public Health - Food Safety: Temporary food service establishment inspections,food_worker_cards,Environmental Public Health - Food Safety: Food service establishment complaints investigated,foodborne_investigations,...,Prevention & Promotion - Tobacco Prevention and Control: Compliance inspections/investigations conducted,Prevention & Promotion - Tobacco Prevention and Control: Citations/fines issued for violations,Prevention & Promotion - Injury Prevention: Is your LHJ involved in initiatives to decrease the rate of injury the community,Other Prevention & Promotion Initiatives: LHJ addressed other health risks through prevention and wellness initiatives - Asthma,Other Prevention & Promotion Initiatives: LHJ addressed other health risks through prevention and wellness initiatives - HIV/AIDS,Other Prevention & Promotion Initiatives: LHJ addressed other health risks through prevention and wellness initiatives - Sexual health/STDs,Other Prevention & Promotion Initiatives: LHJ addressed other health risks through prevention and wellness initiatives - Violence,Other Prevention & Promotion Initiatives: LHJ addressed other health risks through prevention and wellness initiatives - Substance abuse,Other Prevention & Promotion Initiatives: LHJ addressed other health risks through prevention and wellness initiatives - Mental health/ - Behavioral health,Other Prevention & Promotion Initiatives: LHJ addressed other health risks through prevention and wellness initiatives - Land use and/ - or transportation plans
0,Adams County Health Department,125.0,125.0,248.0,148.0,134.0,134.0,917.0,5.0,10.0,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Asotin County Health District,79.0,22.0,28.0,18.0,18.0,18.0,652.0,1.0,1.0,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Benton-Franklin Health District,1409.0,1407.0,3138.0,1036.0,1012.0,1023.0,10116.0,266.0,47.0,...,3.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Chelan-Douglas Health District,710.0,698.0,993.0,438.0,256.0,282.0,6776.0,32.0,22.0,...,8.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Clallam County Department of Health & Human Se...,516.0,516.0,536.0,204.0,177.0,177.0,3256.0,20.0,8.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
